# Splunk Documentation Icons → draw.io Libraries

Builds three diagrams.net / draw.io shape libraries from Splunk's official icon sheet
`Splunk_Documentation_Icons_August2018.png`.

**Outputs**
- `dist/Splunk-Icons-adaptive.xml` — CSS `light-dark()` SVG silhouettes (recommended)
- `dist/Splunk-Icons-color.xml` — full-color PNG icons
- `dist/Splunk-Icons-dark.xml` — inverted PNG icons for dark backgrounds

Place the source PNG at `source/Splunk_Documentation_Icons_August2018.png` or set `DIRECT_PNG_URL`.

In [ ]:
%pip install -q pillow numpy scipy pytesseract nbformat

## 1. Helpers

Encoding, shape XML, data URIs, dark-mode inversion, adaptive SVG silhouettes, and library builder.

In [ ]:
import base64
import difflib
import io
import json
import re
import urllib.parse
import zlib
from pathlib import Path

import numpy as np
from PIL import Image
import pytesseract
from scipy import ndimage

ROOT = Path('.').resolve()
SOURCE_DIR = ROOT / 'source'
DIST_DIR = ROOT / 'dist'
CROPS_DIR = DIST_DIR / 'crops'
SOURCE_DIR.mkdir(exist_ok=True)
DIST_DIR.mkdir(exist_ok=True)
CROPS_DIR.mkdir(exist_ok=True)

SHEET = SOURCE_DIR / 'Splunk_Documentation_Icons_August2018.png'
DIRECT_PNG_URL = ''  # optional: direct URL or JWT-backed CDN link to the PNG


def encode_mx(data: str) -> str:
    """draw.io mxlibrary compression: quote → raw-deflate → base64."""
    quoted = urllib.parse.quote(data, safe='')
    compressor = zlib.compressobj(wbits=-15)
    compressed = compressor.compress(quoted.encode('utf-8')) + compressor.flush()
    return base64.b64encode(compressed).decode('ascii')


def display_size(w: int, h: int, max_size: int = 96) -> tuple[int, int]:
    scale = min(max_size / w, max_size / h, 1.0)
    return max(1, int(w * scale)), max(1, int(h * scale))


def png_data_uri(png_bytes: bytes) -> str:
    return 'data:image/png;base64,' + base64.b64encode(png_bytes).decode('ascii')


def svg_data_uri(svg: str) -> str:
    return 'data:image/svg+xml,' + urllib.parse.quote(svg, safe='')


def invert_for_dark(img: Image.Image) -> Image.Image:
    """Invert RGB where alpha > 10 (keep transparent pixels transparent)."""
    rgba = np.array(img.convert('RGBA'))
    mask = rgba[:, :, 3] > 10
    out = rgba.copy()
    out[mask, 0] = 255 - out[mask, 0]
    out[mask, 1] = 255 - out[mask, 1]
    out[mask, 2] = 255 - out[mask, 2]
    return Image.fromarray(out, 'RGBA')


def silhouette_svg(png_bytes: bytes, w: int, h: int) -> str:
    """SVG silhouette using CSS light-dark() via feFlood + feComposite."""
    b64 = base64.b64encode(png_bytes).decode('ascii')
    return (
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{w}" height="{h}" viewBox="0 0 {w} {h}">'
        '<defs>'
        '<filter id="adaptive" color-interpolation-filters="sRGB">'
        '<feFlood flood-color="light-dark(#2B2B2B,#E8E8E8)" result="flood"/>'
        '<feComposite in="flood" in2="SourceAlpha" operator="in" result="color"/>'
        '<feComposite in="color" in2="SourceGraphic" operator="in"/>'
        '</filter>'
        '</defs>'
        f'<image width="{w}" height="{h}" href="data:image/png;base64,{b64}" filter="url(#adaptive)"/>'
        '</svg>'
    )


def shape_xml(title: str, w: int, h: int, image_data: str, aspect: str = 'fixed') -> dict:
    """mxGraphModel image cell entry for draw.io."""
    dw, dh = display_size(w, h)
    style = (
        f'shape=image;verticalLabelPosition=bottom;verticalAlign=top;'
        f'imageAspect={aspect};image={image_data};'
    )
    return {
        'xml': (
            '<mxGraphModel><root>'
            '<mxCell id="0"/>'
            '<mxCell id="1" parent="0"/>'
            f'<mxCell id="2" value="{title}" style="{style}" vertex="1" parent="1">'
            f'<mxGeometry width="{dw}" height="{dh}" as="geometry"/>'
            '</mxCell>'
            '</root></mxGraphModel>'
        ),
        'w': dw,
        'h': dh,
        'title': title,
        'aspect': aspect,
    }


def build_library(shapes: list[dict]) -> str:
    """Wrap shape dicts in <mxlibrary> with encoded payload."""
    payload = json.dumps(shapes, separators=(',', ':'))
    return '<mxlibrary>' + encode_mx(payload) + '</mxlibrary>'


print('Helpers loaded. ROOT =', ROOT)

## 2. Load source sheet

Use `source/Splunk_Documentation_Icons_August2018.png` or fetch from `DIRECT_PNG_URL`.

In [ ]:
import urllib.request

if DIRECT_PNG_URL:
    print('Fetching', DIRECT_PNG_URL)
    req = urllib.request.Request(DIRECT_PNG_URL, headers={'User-Agent': 'splunk-drawio-icons/1.0'})
    with urllib.request.urlopen(req, timeout=60) as resp:
        sheet_bytes = resp.read()
    sheet = Image.open(io.BytesIO(sheet_bytes)).convert('RGBA')
    SHEET.write_bytes(sheet_bytes)
    print('Saved to', SHEET)
elif SHEET.exists():
    sheet = Image.open(SHEET).convert('RGBA')
    print('Loaded', SHEET, sheet.size)
else:
    raise FileNotFoundError(
        f'Missing {SHEET}. Download Splunk_Documentation_Icons_August2018.png from Splunk docs '
        'or set DIRECT_PNG_URL.'
    )

display(sheet.resize((min(900, sheet.width), int(sheet.height * min(900, sheet.width) / sheet.width))))

## 3. Detect icon boxes

Downsample the alpha mask, label connected components, filter noise, trim crops, and write `dist/crops/` + `manifest.json`.

In [ ]:
def trim_transparent(img: Image.Image, pad: int = 2) -> Image.Image:
    arr = np.array(img)
    alpha = arr[:, :, 3]
    ys, xs = np.where(alpha > 10)
    if len(xs) == 0:
        return img
    x0, x1 = max(0, xs.min() - pad), min(img.width, xs.max() + pad + 1)
    y0, y1 = max(0, ys.min() - pad), min(img.height, ys.max() + pad + 1)
    return img.crop((x0, y0, x1, y1))


def detect_boxes(
    img: Image.Image,
    scale: int = 4,
    min_area: int = 400,
    max_aspect: float = 3.5,
    min_side: int = 12,
    banner_min_width: int = 500,
    banner_max_height: int = 80,
) -> list[dict]:
    rgba = np.array(img)
    alpha = rgba[:, :, 3]
    small = alpha[::scale, ::scale] > 10
    labeled, n = ndimage.label(small)
    boxes = []
    for label_id in range(1, n + 1):
        ys, xs = np.where(labeled == label_id)
        if len(xs) == 0:
            continue
        x0, x1 = xs.min() * scale, (xs.max() + 1) * scale
        y0, y1 = ys.min() * scale, (ys.max() + 1) * scale
        w, h = x1 - x0, y1 - y0
        area = w * h
        if area < min_area:
            continue
        if w < min_side or h < min_side:
            continue
        aspect = max(w / h, h / w)
        if aspect > max_aspect:
            continue
        if w >= banner_min_width and h <= banner_max_height:
            continue
        crop = trim_transparent(img.crop((x0, y0, x1, y1)))
        if crop.width < min_side or crop.height < min_side:
            continue
        boxes.append({
            'x': int(x0), 'y': int(y0),
            'w': crop.width, 'h': crop.height,
            'crop': crop,
        })
    boxes.sort(key=lambda b: (b['y'], b['x']))
    return boxes


boxes = detect_boxes(sheet)
print(f'Detected {len(boxes)} icon candidates')

manifest = []
for i, box in enumerate(boxes):
    fname = f'icon_{i:03d}.png'
    out_path = CROPS_DIR / fname
    box['crop'].save(out_path)
    manifest.append({
        'id': i,
        'file': fname,
        'x': box['x'], 'y': box['y'],
        'w': box['w'], 'h': box['h'],
    })

manifest_path = DIST_DIR / 'manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2))
print('Wrote', manifest_path, 'and', len(manifest), 'crops')

## 4. OCR label bands

Read the text printed below each icon using Tesseract PSM 6 (single uniform block).

In [ ]:
def ocr_label_band(img: Image.Image, box: dict, band_ratio: float = 0.45, pad_x: int = 6) -> str:
    """OCR the region immediately below the icon crop on the full sheet."""
    x, y, w, h = box['x'], box['y'], box['w'], box['h']
    band_h = max(18, int(h * band_ratio))
    x0 = max(0, x - pad_x)
    x1 = min(img.width, x + w + pad_x)
    y0 = min(img.height - 1, y + h)
    y1 = min(img.height, y0 + band_h)
    band = img.crop((x0, y0, x1, y1)).convert('L')
    # boost contrast for OCR
    band_arr = np.array(band)
    band_arr = np.clip((band_arr.astype(np.float32) - 128) * 1.8 + 128, 0, 255).astype(np.uint8)
    band = Image.fromarray(band_arr, 'L')
    text = pytesseract.image_to_string(band, config='--psm 6').strip()
    return re.sub(r'\s+', ' ', text)


ocr_results = []
for i, box in enumerate(boxes):
    raw = ocr_label_band(sheet, box)
    ocr_results.append({'id': i, 'raw_ocr': raw})
    print(f'{i:03d}: {raw!r}')

(DIST_DIR / 'labels_ocr.json').write_text(json.dumps(ocr_results, indent=2))

## 5. Fuzzy-match titles

Seed vocabulary + OCR cleanup + `difflib` matching. Drop bare single letters except `JS`, `HTML`, `CSS`, `SDK` and titles containing *custom visualization*, *simple xml*, or *panels html*.

In [ ]:
KNOWN_TITLES = [
    'Indexer', 'Search Head', 'Heavy Forwarder', 'Universal Forwarder', 'Light Forwarder',
    'Deployment Server', 'License Manager', 'Monitoring Console', 'Cluster Manager',
    'Search Head Cluster Deployer', 'Deployer', 'Indexer Cluster Peer', 'Indexer Cluster',
    'Search Head Cluster Member', 'Search Head Cluster Captain', 'KV Store',
    'App Server', 'Web Server', 'Load Balancer', 'Firewall', 'Router', 'Switch',
    'Database', 'SAN', 'NAS', 'Cloud', 'User', 'Users', 'Admin', 'Developer',
    'Splunk App', 'Splunk Enterprise', 'Splunk Cloud', 'Splunkbase', 'Add-on',
    'Forwarder', 'Intermediate Forwarder', 'Syslog', 'Windows Event Log',
    'Linux', 'Windows', 'Mac', 'Mobile', 'API', 'REST', 'SDK', 'JS', 'HTML', 'CSS',
    'Python', 'Java', 'Script', 'Alert', 'Report', 'Dashboard', 'Data Model',
    'Lookup', 'Field Extraction', 'Tags', 'Event Types', 'Macros', 'Workflow Action',
    'Saved Search', 'Scheduled Search', 'Real-time Search', 'Summary Index',
    'Data Input', 'TCP Input', 'UDP Input', 'HTTP Input', 'Monitor Input',
    'Deployment Client', 'Phone Home', 'Master Node', 'Peer Node', 'Captain',
    'SHC', 'IDX', 'DS', 'LM', 'MC', 'HF', 'UF', 'LF',
    'Custom Visualization', 'Simple XML', 'Panels HTML',
    'Splunk IT Service Intelligence', 'Splunk Enterprise Security',
    'Splunk User', 'Splunk Admin', 'Splunk Developer',
    'Search Peer', 'Search Pool', 'License Slave', 'License Master',
    'Configuration Bundle', 'App Package', 'Server Class',
    'Inputs Data Manager', 'Output', 'Pipeline', 'Queue',
    'Metrics Store', 'Metrics Index', 'Archive', 'Frozen Bucket',
    'Hot Bucket', 'Warm Bucket', 'Cold Bucket', 'SmartStore',
    'S3', 'Azure', 'GCP', 'AWS', 'VM', 'Container', 'Docker', 'Kubernetes',
]

KEEP_BARE = {'js', 'html', 'css', 'sdk'}
KEEP_SUBSTRINGS = ('custom visualization', 'simple xml', 'panels html')


def clean_ocr(text: str) -> str:
    text = text.strip().lower()
    text = re.sub(r'[^a-z0-9+\-/\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def is_bare_letter(text: str) -> bool:
    t = clean_ocr(text)
    if not t:
        return True
    if t in KEEP_BARE:
        return False
    if any(s in t for s in KEEP_SUBSTRINGS):
        return False
    # single token of 1-2 chars that is not in KEEP_BARE
    tokens = t.split()
    if len(tokens) == 1 and len(tokens[0]) <= 2:
        return True
    return False


def fuzzy_title(raw: str, cutoff: float = 0.55) -> str | None:
    cleaned = clean_ocr(raw)
    if not cleaned:
        return None
    # exact case-insensitive match first
    for title in KNOWN_TITLES:
        if clean_ocr(title) == cleaned:
            return title
    matches = difflib.get_close_matches(cleaned, [clean_ocr(t) for t in KNOWN_TITLES], n=1, cutoff=cutoff)
    if matches:
        idx = [clean_ocr(t) for t in KNOWN_TITLES].index(matches[0])
        return KNOWN_TITLES[idx]
    # fallback: title-case the OCR text if it looks reasonable
    if len(cleaned) >= 3 and not is_bare_letter(cleaned):
        return cleaned.title()
    return None


labels_final = []
labels_dropped = []

for entry in ocr_results:
    raw = entry['raw_ocr']
    if is_bare_letter(raw):
        labels_dropped.append({**entry, 'reason': 'bare_letter'})
        continue
    title = fuzzy_title(raw)
    if title is None:
        labels_dropped.append({**entry, 'reason': 'no_match'})
        continue
    labels_final.append({
        'id': entry['id'],
        'raw_ocr': raw,
        'title': title,
        'file': manifest[entry['id']]['file'],
    })

(DIST_DIR / 'labels_final.json').write_text(json.dumps(labels_final, indent=2))
(DIST_DIR / 'labels_dropped.json').write_text(json.dumps(labels_dropped, indent=2))
print(f'Kept {len(labels_final)}, dropped {len(labels_dropped)}')

## 6. Build draw.io libraries

Emit adaptive (SVG), color, and dark XML libraries. Duplicate titles get a `(2)` suffix.

In [ ]:
def uniquify_titles(items: list[dict]) -> list[dict]:
    seen: dict[str, int] = {}
    out = []
    for item in items:
        title = item['title']
        count = seen.get(title, 0) + 1
        seen[title] = count
        final_title = title if count == 1 else f'{title} ({count})'
        out.append({**item, 'title': final_title})
    return out


labeled = uniquify_titles(labels_final)

color_shapes = []
dark_shapes = []
adaptive_shapes = []

for item in labeled:
    crop_path = CROPS_DIR / item['file']
    crop = Image.open(crop_path).convert('RGBA')
    w, h = crop.size
    png_bytes = io.BytesIO()
    crop.save(png_bytes, format='PNG')
    png_bytes = png_bytes.getvalue()

    dark_crop = invert_for_dark(crop)
    dark_buf = io.BytesIO()
    dark_crop.save(dark_buf, format='PNG')
    dark_bytes = dark_buf.getvalue()

    dw, dh = display_size(w, h)
    svg = silhouette_svg(png_bytes, dw, dh)

    color_shapes.append(shape_xml(item['title'], w, h, png_data_uri(png_bytes)))
    dark_shapes.append(shape_xml(item['title'], w, h, png_data_uri(dark_bytes)))
    adaptive_shapes.append(shape_xml(item['title'], w, h, svg_data_uri(svg)))

outputs = {
    DIST_DIR / 'Splunk-Icons-color.xml': build_library(color_shapes),
    DIST_DIR / 'Splunk-Icons-dark.xml': build_library(dark_shapes),
    DIST_DIR / 'Splunk-Icons-adaptive.xml': build_library(adaptive_shapes),
}

for path, content in outputs.items():
    path.write_text(content, encoding='utf-8')
    print(f'Wrote {path} ({len(content):,} bytes, {len(labeled)} shapes)')

print('\nDone! Import in draw.io: File → Open Library From → Device')